# KDE and Density Estimation

**DS4DH Practice Pack · Module 10 — Data Visualization and Communication**

*Technique:* Kernel density estimation, bandwidth choice, and boundary artefacts

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/ds4dh-colab-pack/blob/main/notebooks/10a_kde.ipynb)

Data: `merged_dataset.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# This notebook reads the CSVs sitting next to it. In Colab you will be asked
# to upload them from the pack's data/ folder.
NEEDED = ['merged_dataset.csv']

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('merged_dataset.csv')
CITIES = ['Montréal', 'Toronto', 'Edmonton', 'Vancouver']

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

A histogram's appearance depends on where you put the bin edges — shift them
slightly and a bump can appear or vanish. A **kernel density estimate** removes
that arbitrariness by placing a small smooth curve over every observation and
adding them up.

It replaces the bin-edge choice with a **bandwidth** choice, which is a better
trade but not a free one. This notebook is about making that choice visible.

In [ ]:
# One row per Census Subdivision.
#   • rows with no csd_code are CMA-level and Canada-level aggregates, not CSDs
#   • each CSD appears 3x (Immigrant / Non-immigrants / Total Immigrant Status)
# Keeping either would silently double- or triple-count places.
csd = df.dropna(subset=['csd_code'])
base = csd[(csd['immigrant_status'] == 'Total Immigrant Status')
           & (csd['cma'].isin(CITIES))].copy()

print(f'{len(df):>4} rows in the raw file')
print(f'{len(csd):>4} after dropping CMA/Canada aggregate rows')
print(f'{len(base):>4} CSDs in the four cities (one row each)')

In [ ]:
d = base.dropna(subset=['Total'])
x = d['Total'].values

grid = np.linspace(x.min() - 3, x.max() + 3, 400)
kde = stats.gaussian_kde(x)          # Scott's rule bandwidth by default

fig, ax = plt.subplots()
ax.hist(x, bins=25, density=True, alpha=0.35, edgecolor='white',
        linewidth=0.5, label='histogram')
ax.plot(grid, kde(grid), lw=2.5, color='#E8663D', label="KDE (Scott's rule)")
ax.axvline(30, color='#888', ls=':', label='30% affordability line')
ax.set_xlabel('Total STIR (%)')
ax.set_ylabel('density')
ax.set_title(f'Housing burden across {len(d)} CSDs')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Scott's rule bandwidth factor: {kde.factor:.4f}")

## Bandwidth is the whole argument

Too small and you plot noise as if it were structure. Too large and you smooth
real features away. There is no correct answer — only a defensible one.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, bw, label in zip(axes, [0.15, None, 0.9],
                         ['bw=0.15 (under-smoothed)', "Scott's rule", 'bw=0.9 (over-smoothed)']):
    k = stats.gaussian_kde(x, bw_method=bw)
    ax.plot(grid, k(grid), lw=2, color='#E8663D')
    ax.hist(x, bins=25, density=True, alpha=0.25, edgecolor='white', linewidth=0.4)
    ax.set_title(label)
    ax.set_xlabel('Total STIR (%)')
plt.tight_layout()
plt.show()

print('Left: every wiggle is one or two municipalities.')
print('Right: a single hump that hides whatever structure exists.')

### 🔧 Your turn 1

Change the middle panel's `bw_method` to `'silverman'`.

How different is it from Scott's rule on this data? Both are automatic rules
based on sample size and spread — when would you expect them to disagree
substantially?

## The boundary artefact

STIR cannot be negative, but a Gaussian kernel does not know that. Placed over an
observation near zero, half its mass falls into impossible territory — so the KDE
assigns visible density to values that cannot occur, and understates density at
the boundary itself.

In [ ]:
low = d['Total'].min()
below = kde(np.linspace(-5, 0, 50))

print(f'lowest observed STIR: {low:.1f}%')
print(f'KDE density assigned below 0%: {below.max():.5f}')
print()
if below.max() > 1e-6:
    print('The curve leaks past zero. On a variable with a hard floor, either')
    print('truncate the plotted range or use a reflected/boundary-corrected KDE.')
else:
    print('Negligible here — the data sits far from the boundary.')

In [ ]:
# Reflection: mirror the data at the boundary, estimate, then double and clip.
def reflected_kde(values, boundary=0.0, bw=None):
    mirrored = np.concatenate([values, 2 * boundary - values])
    return stats.gaussian_kde(mirrored, bw_method=bw)

rk = reflected_kde(x)
g2 = np.linspace(0, x.max() + 3, 400)

fig, ax = plt.subplots()
ax.plot(g2, kde(g2), lw=2, label='plain KDE')
ax.plot(g2, rk(g2) * 2, lw=2, ls='--', color='#E8663D', label='reflected at 0')
ax.set_xlabel('Total STIR (%)')
ax.set_ylabel('density')
ax.set_title('Boundary correction (the difference is at the left edge)')
ax.legend()
plt.tight_layout()
plt.show()

## Comparing groups with KDE

This is where KDE earns its place over histograms: overlaid densities compare
cleanly, where overlaid histograms become unreadable at three or more groups.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
for city in CITIES:
    s = d[d['cma'] == city]['Total'].values
    if len(s) < 5:
        continue
    k = stats.gaussian_kde(s)
    ax.plot(grid, k(grid), lw=2, label=f'{city} (n={len(s)})')
ax.axvline(30, color='#888', ls=':', label='30% line')
ax.set_xlabel('Total STIR (%)')
ax.set_ylabel('density')
ax.set_title('Housing burden by metropolitan area')
ax.legend()
plt.tight_layout()
plt.show()

print('Each curve integrates to 1, so this compares SHAPES, not counts.')
print('A city with 22 CSDs looks as tall as one with 88 — say so in the caption.')

### 🔧 Your turn 2

The curves above hide that the four cities have very different sample sizes.

Rescale each curve by its `n` so the areas are proportional to the number of
CSDs. Which version answers "where is burden concentrated?" and which answers
"what does a typical municipality in each city look like?"

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

**Your turn 1.** Silverman and Scott give very similar bandwidths for
approximately normal data and diverge for multimodal or heavily skewed
distributions — Silverman tends to over-smooth in those cases, because it assumes
normality more strongly. With 155 moderately skewed observations they land close
together here. Where two automatic rules disagree noticeably, that disagreement is
itself a signal that the distribution has structure worth looking at directly.

**Your turn 2.** The equal-area version answers "what does a typical municipality
in each city look like" — it is a shape comparison. The n-scaled version answers
"where is burden concentrated" — it lets Montréal's 88 CSDs dominate visually,
which is correct if your question is about the stock of high-burden places rather
than about the character of each city. Choosing between them is choosing your
question, and the caption should say which.

</details>

## Where this stops

You can show a distribution honestly. The next notebook is about the chart type
that actually persuades: a ranked bar.